# Correlation Heatmap
Load a combined compare JSON file and render only its market correlation heatmap.

In [ ]:
# 1. ESSENTIAL INSTALLATION (Must run every time you refresh)
try:
    import piplite
    await piplite.install(["seaborn", "pandas"])
except ImportError:
    pass

import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import clear_output

# 2. CONFIGURATION
plt.style.use("seaborn-v0_8-whitegrid")
BASE_DIR = Path.cwd()
SUPPORTED_SCHEMA_TYPE = "polymarket_combined_compare"

def plot_file(filename):
    path = BASE_DIR / filename
    try:
        payload = json.loads(path.read_text())
        rows = []
        for series in payload["series"]:
            for pt in series["points"]:
                rows.append({
                    "market": series["name"],
                    "timestamp": pd.to_datetime(pt["t"], unit="s", utc=True),
                    "price": float(pt["p"]),
                })

        df = pd.DataFrame(rows).pivot_table(
            index="timestamp", columns="market", values="price", aggfunc="last"
        ).sort_index().interpolate(method="time").ffill().bfill()

        plt.figure(figsize=(10, 8))
        sns.heatmap(df.corr(), annot=True, cmap="Spectral", center=0)
        plt.title(f"Correlations: {path.stem.upper()}")
        plt.show()
    except Exception as e:
        print(f"Error processing {filename}: {e}")

# 3. INTERACTIVE SELECTION
files = sorted([p.name for p in BASE_DIR.glob("*.json")])

if not files:
    print(f"No JSON files found in {BASE_DIR}. Please upload files to the environment.")
else:
    print("--- Available Data Files ---")
    for i, f in enumerate(files):
        print(f"[{i}] {f}")
    
    try:
        selection = input("\nSelect file index (e.g., 0) or 'q' to quit: ")
        if selection.lower() != 'q':
            idx = int(selection)
            if 0 <= idx < len(files):
                clear_output()
                print(f"Generating heatmap for: {files[idx]}...")
                plot_file(files[idx])
            else:
                print("Index out of range.")
    except ValueError:
        print("Please enter a valid number.")